# Exercise 1: Content-Based Filtering
In this exercise, we will implement a simple content-based filtering system using a movie dataset.
The goal is to recommend movies based on their genres.


We want to recommend movies to a user who has shown in interest in The Godfather.

In [1]:
import pandas as pd
import numpy as np

# Sample movie dataset
movies = pd.DataFrame({
    'movie_id': [1, 2, 3, 4, 5],
    'title': ['The Matrix', 'The Godfather', 'Toy Story', 'The Dark Knight', 'Pulp Fiction'],
    'genres': ['Action|Sci-Fi', 'Crime|Drama', 'Animation|Children|Comedy', 'Action|Crime|Drama', 'Crime|Drama']
})


Step 1: Encode the genres using TF-IDF vectorization.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create a TF-IDF vectorizer
vectorizer = TfidfVectorizer(tokenizer=lambda x: x.split('|'), token_pattern=None, lowercase=False)
# Fit and transform the genres
tfidf_matrix = vectorizer.fit_transform(movies['genres'])
print(vectorizer.get_feature_names_out()) # Check the feature names
print(tfidf_matrix.shape)  # Check the shape of the TF-IDF matrix, does it make sense?
print(tfidf_matrix.toarray())  # Display the TF-IDF matrix, does it make sense?

['Action' 'Animation' 'Children' 'Comedy' 'Crime' 'Drama' 'Sci-Fi']
(5, 7)
[[0.62791376 0.         0.         0.         0.         0.
  0.77828292]
 [0.         0.         0.         0.         0.70710678 0.70710678
  0.        ]
 [0.         0.57735027 0.57735027 0.57735027 0.         0.
  0.        ]
 [0.64846263 0.         0.         0.         0.53828256 0.53828256
  0.        ]
 [0.         0.         0.         0.         0.70710678 0.70710678
  0.        ]]


c:\Users\michiela\AppData\Local\miniconda3\envs\DS\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Look at the TF-IDF matrix, does it make sense? Think about the genres and how common they are across movies. Write down your observations below, mention specific values.

**ANSWER:**

The matrix makes sense. Each column represents a genre and each row a movie. Genres that occur in fewer movies receive a larger IDF weight. For example, Sci-Fi occurs only in The Matrix, so it receives a relatively large weight. Crime and Drama occur in three movies, so their weights are lower. The Godfather and Pulp Fiction have exactly the same two genres, so their TF-IDF vectors are identical.

Step 2: Compute the cosine similarity between the movies based on their genres.

In [3]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix)

# Create a DataFrame with movie titles as index and columns
cosine_sim_df = pd.DataFrame(cosine_sim, index=movies['title'], columns=movies['title'])
cosine_sim_df

title,The Matrix,The Godfather,Toy Story,The Dark Knight,Pulp Fiction
title,,,,,
The Matrix,1.000000,0.000000,0.0,0.407179,0.000000
The Godfather,0.000000,1.000000,0.0,0.761246,1.000000
Toy Story,0.000000,0.000000,1.0,0.000000,0.000000
The Dark Knight,0.407179,0.761246,0.0,1.000000,0.761246
Pulp Fiction,0.000000,1.000000,0.0,0.761246,1.000000


Interpret the cosine similarity matrix. What does it tell you about the similarity between movies based on their genres? does it make sense? Write down your observations below.

**ANSWER:**

A cosine similarity of 1 means identical genre profiles, while 0 means that the movies share no genres. The Godfather and Pulp Fiction have similarity 1.000 because both are exactly Crime|Drama. The Godfather and The Dark Knight have similarity about 0.761 because they share Crime and Drama, but The Dark Knight also has Action. The Matrix and The Dark Knight have a lower similarity of about 0.407 because they only share Action.

So, which movie would you recommend to a user who liked "The Godfather"?

Pulp Fiction, because it has the highest cosine similarity with The Godfather (1.000), excluding The Godfather itself.

### Exercise 2: Collaborative Filtering: User-User Similarity (User-Based)

In this exercise, we will implement a user-based collaborative filtering system using a movie rating dataset.

Before we start, let's understand the dataset. We have a set of users and their ratings for various movies. We want to determine what rating Dave would give to "Inception" based on the ratings of other users.

In [4]:
ratings = [
    {'UserName': 'Alice', 'MovieName': 'Inception', 'Rating': 5},
    {'UserName': 'Alice', 'MovieName': 'Titanic', 'Rating': 3},
    {'UserName': 'Alice', 'MovieName': 'Avatar', 'Rating': 4},
    {'UserName': 'Alice', 'MovieName': 'Matrix', 'Rating': 5},
    {'UserName': 'Alice', 'MovieName': 'Joker', 'Rating': 3},
    {'UserName': 'Alice', 'MovieName': 'Up', 'Rating': 4},
    {'UserName': 'Bob', 'MovieName': 'Inception', 'Rating': 3},
    {'UserName': 'Bob', 'MovieName': 'Titanic', 'Rating': 4},
    {'UserName': 'Bob', 'MovieName': 'Avatar', 'Rating': 2},
    {'UserName': 'Bob', 'MovieName': 'Matrix', 'Rating': 2},
    {'UserName': 'Bob', 'MovieName': 'Joker', 'Rating': 5},
    {'UserName': 'Bob', 'MovieName': 'Up', 'Rating': 3},
    {'UserName': 'Carol', 'MovieName': 'Inception', 'Rating': 4},
    {'UserName': 'Carol', 'MovieName': 'Titanic', 'Rating': 2},
    {'UserName': 'Carol', 'MovieName': 'Avatar', 'Rating': 5},
    {'UserName': 'Carol', 'MovieName': 'Matrix', 'Rating': 4},
    {'UserName': 'Carol', 'MovieName': 'Joker', 'Rating': 2},
    {'UserName': 'Carol', 'MovieName': 'Up', 'Rating': 5},
    {'UserName': 'Dave', 'MovieName': 'Titanic', 'Rating': 5},
    {'UserName': 'Dave', 'MovieName': 'Avatar', 'Rating': 3},
    {'UserName': 'Dave', 'MovieName': 'Matrix', 'Rating': 3},
    {'UserName': 'Dave', 'MovieName': 'Joker', 'Rating': 4},
    {'UserName': 'Dave', 'MovieName': 'Up', 'Rating': 2},
    {'UserName': 'Eve', 'MovieName': 'Inception', 'Rating': 2},
    {'UserName': 'Eve', 'MovieName': 'Titanic', 'Rating': 3},
    {'UserName': 'Eve', 'MovieName': 'Avatar', 'Rating': 1},
    {'UserName': 'Eve', 'MovieName': 'Matrix', 'Rating': 1},
    {'UserName': 'Eve', 'MovieName': 'Joker', 'Rating': 5},
    {'UserName': 'Eve', 'MovieName': 'Up', 'Rating': 2},
    {'UserName': 'Frank', 'MovieName': 'Inception', 'Rating': 5},
    {'UserName': 'Frank', 'MovieName': 'Titanic', 'Rating': 4},
    {'UserName': 'Frank', 'MovieName': 'Avatar', 'Rating': 5},
    {'UserName': 'Frank', 'MovieName': 'Matrix', 'Rating': 5},
    {'UserName': 'Frank', 'MovieName': 'Joker', 'Rating': 3},
    {'UserName': 'Frank', 'MovieName': 'Up', 'Rating': 5}
]


Step 1: Convert this data into a format suitable for analysis. We will create a DataFrame from the ratings dictionary. Additionally, we will pivot the DataFrame to create a user-item matrix where rows are movies and columns are users. The values will be the ratings given by users to movies. This will help us in computing similarity between users.

In [5]:
ratings_df = pd.DataFrame(ratings)
ratings_df = ratings_df.pivot(index='MovieName', columns='UserName', values='Rating')
ratings_df

UserName,Alice,Bob,Carol,Dave,Eve,Frank
MovieName,,,,,,
Avatar,4.0,2.0,5.0,3.0,1.0,5.0
Inception,5.0,3.0,4.0,NaN,2.0,5.0
Joker,3.0,5.0,2.0,4.0,5.0,3.0
Matrix,5.0,2.0,4.0,3.0,1.0,5.0
Titanic,3.0,4.0,2.0,5.0,3.0,4.0
Up,4.0,3.0,5.0,2.0,2.0,5.0


Step 2: Compute the Pearson correlation coefficient between users to find similar users. Note, there are multiple ways to do this; the simplest is to use the `corr` method in pandas. Pay attention to whether the correlation is computed across rows or columns. What do we need here?

In [6]:
# Compute the Pearson correlation coefficient
user_similarity = ratings_df.corr(method='pearson')
np.fill_diagonal(user_similarity.values, 0.0) # Set diagonal to zero to avoid self-similarity
user_similarity = pd.DataFrame(user_similarity, index=ratings_df.columns, columns=ratings_df.columns)
user_similarity

UserName,Alice,Bob,Carol,Dave,Eve,Frank
UserName,,,,,,
Alice,0.000000,-0.765092,0.654654,-0.681385,-0.742611,0.801784
Bob,-0.765092,0.000000,-0.834784,0.605406,0.984820,-0.920158
Carol,0.654654,-0.834784,0.000000,-0.896388,-0.810255,0.874818
Dave,-0.681385,0.605406,-0.896388,0.000000,0.550350,-0.686406
Eve,-0.742611,0.984820,-0.810255,0.550350,0.000000,-0.952661
Frank,0.801784,-0.920158,0.874818,-0.686406,-0.952661,0.000000


Based on the user similarity matrix, which users are most similar to Dave? Write down your observations below.

**ANSWER:**

The users most similar to Dave are Bob (Pearson correlation about 0.605) and Eve (about 0.550). Alice, Frank, and Carol have negative correlations with Dave, which means their rating patterns move in the opposite direction on the movies they have both rated. In this simplified neighborhood model, we will use only positively similar users.

Now, we will use the similarity scores to predict Dave's rating for "Inception". We will use a weighted average of the ratings from similar users, weighted by their similarity to Dave. Try to make this a modular function that takes the user similarity matrix, the ratings DataFrame, the user for whom we want to predict the rating and the movie for which we want to predict the rating. Use your Pandas knowledge to filter the DataFrame and compute the weighted average. Only consider users who have rated the movie and have a non-negative similarity score with Dave.
(Note: To be theoretically sound, we would center by user means when using Pearson. That would take us too far here, the values probably seem overinflated now)

In [7]:
def predict_rating(user_similarity, ratings_df, target_user, target_movie, min_sim = 0.0):
    # Get the similarity scores for the target user and drop the entry for the target user
    sim_scores = user_similarity.loc[target_user]
    sim_scores = sim_scores[sim_scores > min_sim]
    print(f"Similarity scores for {target_user}:\n{sim_scores}\n")
    # Get the ratings of the target movie from other users
    movie_ratings = ratings_df.loc[target_movie].dropna()
    print(f"Ratings for {target_movie}:\n{movie_ratings}\n")

    sim_scores = sim_scores.reindex(movie_ratings.index).dropna()
    if sim_scores.empty:
        return None

    predicted_rating = (movie_ratings * sim_scores).sum() / abs(sim_scores).sum()

    return predicted_rating

In [8]:
# Predict Dave's rating for "Inception"
predicted_rating = predict_rating(user_similarity, ratings_df, 'Dave', 'Inception')
print(f"Predicted rating for 'Inception' by Dave: {predicted_rating:.2f}")

Similarity scores for Dave:
UserName
Bob    0.605406
Eve    0.550350
Name: Dave, dtype: float64

Ratings for Inception:
UserName
Alice    5.0
Bob      3.0
Carol    4.0
Eve      2.0
Frank    5.0
Name: Inception, dtype: float64

Predicted rating for 'Inception' by Dave: 2.52
